In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 04. Model Improvement

## Objective

This notebook evaluates whether additional historical demand features can
improve the existing demand forecasting model.

The current forecasting model is used as the benchmark.

### Existing benchmark

- Model: HistGradientBoosting
- Test MAE: 8.9771
- Test RMSE: 11.0677
- Test WAPE: 33.86%

The improvement experiment will introduce historical demand features
and evaluate whether they provide better out-of-sample forecasting
performance.

The existing model and its results will be preserved as the benchmark.

In [4]:
# Menentukan folder utama project ForecastOpti.
PROJECT_DIR = "/content/drive/MyDrive/ForecastOpti"

# Menampilkan seluruh file CSV yang sudah ada di project.
# Kita hanya mencari dataset hasil tahap sebelumnya;
# tidak melakukan cleaning atau preprocessing ulang.
import os

for root, dirs, files in os.walk(PROJECT_DIR):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))


/content/drive/MyDrive/ForecastOpti/data/raw/retail_sales.csv
/content/drive/MyDrive/ForecastOpti/outputs/forecasts/test_forecast.csv
/content/drive/MyDrive/ForecastOpti/outputs/forecasts/daily_forecast_summary.csv
/content/drive/MyDrive/ForecastOpti/outputs/evaluation/test_evaluation.csv


In [5]:
import pandas as pd

RAW_PATH = "/content/drive/MyDrive/ForecastOpti/data/raw/retail_sales.csv"

# Hanya membaca dataset raw.
# Tidak melakukan cleaning, filtering, atau perubahan data.
raw_df = pd.read_csv(RAW_PATH)

print("Shape:", raw_df.shape)
print("\nColumns:")
print(raw_df.columns.tolist())

display(raw_df.head())

Shape: (4565000, 8)

Columns:
['date', 'store_id', 'item_id', 'sales', 'price', 'promo', 'weekday', 'month']


,date,store_id,item_id,sales,price,promo,weekday,month
0,2019-01-01,store_1,item_1,41,21.30,0,1,1
1,2019-01-02,store_1,item_1,53,21.30,0,2,1
2,2019-01-03,store_1,item_1,39,21.30,0,3,1
3,2019-01-04,store_1,item_1,35,21.30,0,4,1
4,2019-01-05,store_1,item_1,51,17.04,1,5,1


In [7]:
# Menggunakan dataset raw yang sudah berhasil dimuat sebelumnya.
# Tidak menggunakan variabel df dari notebook lama.
model_df = raw_df.copy()

# Memastikan kolom tanggal bertipe datetime.
model_df["date"] = pd.to_datetime(model_df["date"])

# Mengurutkan data secara temporal untuk setiap store-item.
model_df = model_df.sort_values(
    ["store_id", "item_id", "date"]
).reset_index(drop=True)

# Mendefinisikan unit time series.
group_cols = ["store_id", "item_id"]

# Membuat fitur demand historis.
model_df["sales_lag_1"] = (
    model_df.groupby(group_cols)["sales"].shift(1)
)

model_df["sales_lag_7"] = (
    model_df.groupby(group_cols)["sales"].shift(7)
)

model_df["sales_lag_14"] = (
    model_df.groupby(group_cols)["sales"].shift(14)
)

# Memeriksa hasil lag features.
display(
    model_df[
        [
            "date",
            "store_id",
            "item_id",
            "sales",
            "sales_lag_1",
            "sales_lag_7",
            "sales_lag_14"
        ]
    ].head(20)
)

,date,store_id,item_id,sales,sales_lag_1,sales_lag_7,sales_lag_14
0,2019-01-01,store_1,item_1,41,NaN,NaN,NaN
1,2019-01-02,store_1,item_1,53,41.0,NaN,NaN
2,2019-01-03,store_1,item_1,39,53.0,NaN,NaN
3,2019-01-04,store_1,item_1,35,39.0,NaN,NaN
4,2019-01-05,store_1,item_1,51,35.0,NaN,NaN
5,2019-01-06,store_1,item_1,38,51.0,NaN,NaN
6,2019-01-07,store_1,item_1,45,38.0,NaN,NaN
7,2019-01-08,store_1,item_1,48,45.0,41.0,NaN
8,2019-01-09,store_1,item_1,50,48.0,53.0,NaN
9,2019-01-10,store_1,item_1,44,50.0,39.0,NaN


In [8]:
# Menentukan fitur yang membutuhkan histori.
lag_features = [
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_14"
]

# Hanya baris yang memiliki seluruh lag yang diperlukan
# yang digunakan untuk eksperimen model improvement.
model_df_improved = model_df.dropna(
    subset=lag_features
).copy()

# Memastikan tidak ada missing value pada lag features.
print("Rows before removing incomplete history :", len(model_df))
print("Rows after removing incomplete history  :", len(model_df_improved))

print("\nMissing values in lag features:")
print(
    model_df_improved[lag_features].isna().sum()
)

Rows before removing incomplete history : 4565000
Rows after removing incomplete history  : 4530000

Missing values in lag features:
sales_lag_1     0
sales_lag_7     0
sales_lag_14    0
dtype: int64


In [9]:
# Fitur yang sudah digunakan oleh model sebelumnya.
base_features = [
    "price",
    "promo",
    "weekday",
    "month"
]

# Fitur historical demand yang baru ditambahkan.
improved_features = base_features + [
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_14"
]

# Menampilkan feature set yang akan digunakan.
print("Improved model features:")
print(improved_features)

print("\nNumber of features:", len(improved_features))

Improved model features:
['price', 'promo', 'weekday', 'month', 'sales_lag_1', 'sales_lag_7', 'sales_lag_14']

Number of features: 7


In [10]:
# Menggunakan tanggal yang sama dengan eksperimen forecasting sebelumnya.
train_end = "2022-12-31"
validation_end = "2023-06-30"

# Membagi dataset berdasarkan waktu.
# Tidak menggunakan random split karena ini adalah forecasting.
train_improved = model_df_improved[
    model_df_improved["date"] <= train_end
].copy()

validation_improved = model_df_improved[
    (model_df_improved["date"] > train_end) &
    (model_df_improved["date"] <= validation_end)
].copy()

test_improved = model_df_improved[
    model_df_improved["date"] > validation_end
].copy()

# Memisahkan features dan target.
X_train_improved = train_improved[improved_features]
y_train_improved = train_improved["sales"]

X_validation_improved = validation_improved[improved_features]
y_validation_improved = validation_improved["sales"]

X_test_improved = test_improved[improved_features]
y_test_improved = test_improved["sales"]

# Menampilkan ukuran masing-masing dataset.
print("X_train_improved     :", X_train_improved.shape)
print("y_train_improved     :", y_train_improved.shape)

print("X_validation_improved:", X_validation_improved.shape)
print("y_validation_improved:", y_validation_improved.shape)

print("X_test_improved      :", X_test_improved.shape)
print("y_test_improved      :", y_test_improved.shape)

X_train_improved     : (3617500, 7)
y_train_improved     : (3617500,)
X_validation_improved: (452500, 7)
y_validation_improved: (452500,)
X_test_improved      : (460000, 7)
y_test_improved      : (460000,)


In [11]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Membuat model dengan algoritma yang sama seperti model terbaik sebelumnya.
# Perbedaannya hanya pada feature set:
# model baru menggunakan 4 fitur lama + 3 lag features.
improved_model = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.08,
    max_leaf_nodes=31,
    random_state=42
)

# Training hanya menggunakan data sampai 2022-12-31.
# Validation dan test tetap tidak digunakan selama training.
improved_model.fit(
    X_train_improved,
    y_train_improved
)

print("Improved HistGradientBoosting training completed.")

Improved HistGradientBoosting training completed.


In [12]:
# Membuat prediksi pada validation set.
# Validation digunakan untuk menilai apakah fitur lag
# memberikan improvement sebelum melihat hasil test.
y_validation_pred_improved = improved_model.predict(
    X_validation_improved
)

# Menghitung MAE.
validation_mae_improved = mean_absolute_error(
    y_validation_improved,
    y_validation_pred_improved
)

# Menghitung RMSE.
validation_rmse_improved = np.sqrt(
    mean_squared_error(
        y_validation_improved,
        y_validation_pred_improved
    )
)

print(f"Improved Validation MAE  : {validation_mae_improved:.4f}")
print(f"Improved Validation RMSE : {validation_rmse_improved:.4f}")

Improved Validation MAE  : 3.0177
Improved Validation RMSE : 3.8302


In [13]:
# Membuat prediksi pada test set yang belum digunakan
# selama proses training maupun pemilihan model.
y_test_pred_improved = improved_model.predict(
    X_test_improved
)

# Menghitung Test MAE.
test_mae_improved = mean_absolute_error(
    y_test_improved,
    y_test_pred_improved
)

# Menghitung Test RMSE.
test_rmse_improved = np.sqrt(
    mean_squared_error(
        y_test_improved,
        y_test_pred_improved
    )
)

print(f"Improved Test MAE  : {test_mae_improved:.4f}")
print(f"Improved Test RMSE : {test_rmse_improved:.4f}")

Improved Test MAE  : 2.9689
Improved Test RMSE : 3.7430


In [14]:
# Menghitung Weighted Absolute Percentage Error (WAPE)
# untuk improved model pada test set.

test_wape_improved = (
    np.sum(
        np.abs(
            y_test_improved - y_test_pred_improved
        )
    )
    / np.sum(
        np.abs(y_test_improved)
    )
) * 100

print(f"Improved Test WAPE : {test_wape_improved:.2f}%")

Improved Test WAPE : 11.20%


In [15]:
improvement_summary = pd.DataFrame({
    "model": [
        "HistGradientBoosting",
        "HistGradientBoosting + Lag Features"
    ],
    "test_mae": [
        8.9771,
        test_mae_improved
    ],
    "test_rmse": [
        11.0677,
        test_rmse_improved
    ],
    "test_wape": [
        33.86,
        test_wape_improved
    ]
})

# Menghitung improvement percentage
improvement_summary["mae_improvement_pct"] = (
    (
        improvement_summary.loc[0, "test_mae"]
        - improvement_summary["test_mae"]
    )
    / improvement_summary.loc[0, "test_mae"]
    * 100
)

improvement_summary["rmse_improvement_pct"] = (
    (
        improvement_summary.loc[0, "test_rmse"]
        - improvement_summary["test_rmse"]
    )
    / improvement_summary.loc[0, "test_rmse"]
    * 100
)

improvement_summary["wape_improvement_pct"] = (
    (
        improvement_summary.loc[0, "test_wape"]
        - improvement_summary["test_wape"]
    )
    / improvement_summary.loc[0, "test_wape"]
    * 100
)

display(improvement_summary)

,model,test_mae,test_rmse,test_wape,mae_improvement_pct,rmse_improvement_pct,wape_improvement_pct
0,HistGradientBoosting,8.977100,11.067700,33.860000,0.000000,0.000000,0.000000
1,HistGradientBoosting + Lag Features,2.968901,3.742962,11.199173,66.928058,66.181213,66.925064


In [16]:
import os
import joblib

# Folder penyimpanan model
MODEL_DIR = "/content/drive/MyDrive/ForecastOpti/models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Path model improved
improved_model_path = os.path.join(
    MODEL_DIR,
    "hist_gradient_boosting_lag.pkl"
)

# Simpan model
joblib.dump(
    improved_model,
    improved_model_path
)

print("Improved model saved to:")
print(improved_model_path)

Improved model saved to:
/content/drive/MyDrive/ForecastOpti/models/hist_gradient_boosting_lag.pkl


In [17]:
import os

EVAL_DIR = "/content/drive/MyDrive/ForecastOpti/outputs/evaluation"
os.makedirs(EVAL_DIR, exist_ok=True)

improvement_path = os.path.join(
    EVAL_DIR,
    "model_improvement_comparison.csv"
)

improvement_summary.to_csv(
    improvement_path,
    index=False
)

print("Improvement evaluation saved to:")
print(improvement_path)

Improvement evaluation saved to:
/content/drive/MyDrive/ForecastOpti/outputs/evaluation/model_improvement_comparison.csv
